In [1]:
from model.loss_function.siamese.custom_siamese_loss import CustomSiameseLoss
from model.architecture.siamese_model import SiameseAHPModel
from data_augmentation.matrices_loader import MatricesLoader
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from data_augmentation.datasets.siamese_AHP_matrix_dataset import SiameseAHPMatrixDataset
from data_augmentation.generator.target_cr_generator import TargetCRMatricesGenerator

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device used: cpu


In [3]:
model = SiameseAHPModel(n_criteria=5).to(device)
loader = MatricesLoader()
matrices, weights = loader.load_noisy_cr_matrices(0.10, False)
generator = TargetCRMatricesGenerator(matrices.shape[0], matrices.shape[1], 0.10)
comparison_matrices = generator.from_weights(weights)

dataset = SiameseAHPMatrixDataset(matrices, comparison_matrices, weights)


Successfuly uploaded 10 noised matrices (level: c010).


In [ ]:
def train_model(model, dataset, epochs=100, batch_size=32, lr=0.001):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    criterion = CustomSiameseLoss(lambda_cop=1.0, lambda_rec=0.5, lambda_stab=0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = []
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, (m1_batch, m2_batch, weights_batch) in enumerate(dataloader):
            m1_batch = m1_batch.to(device)
            m2_batch = m2_batch.to(device)
            weights_batch = weights_batch.to(device)

            logits1, logits2 = model(m1_batch, m2_batch)
            loss = criterion(logits1, logits2, weights_batch, m1_batch.squeeze(1), m2_batch.squeeze(1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        history.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoka [{epoch+1}/{epochs}] | Średnia Strata: {avg_loss:.6f}")

    print("Trening zakończony.")
    return model, history

In [7]:
trained_model, loss_history = train_model(model, dataset)

tensor([[-0.0418, -0.1836, -0.3320,  0.0331,  0.2606],
        [-0.0168, -0.1174, -0.1924,  0.0308,  0.1994],
        [-0.0421, -0.1586, -0.2510,  0.1301,  0.1005],
        [-0.0213, -0.1582, -0.2723,  0.0569,  0.1972],
        [-0.0180, -0.1593, -0.3163,  0.0812,  0.1684],
        [-0.0298, -0.1327, -0.2744,  0.1084,  0.0772],
        [ 0.2013, -0.1583, -0.3541, -0.0649,  0.1670],
        [-0.0221, -0.1402, -0.2345,  0.1039,  0.0876],
        [-0.0124, -0.1061, -0.3201,  0.0922,  0.2314],
        [-0.0113, -0.1413, -0.1826,  0.0147,  0.2481]],
       grad_fn=<AddmmBackward0>)
tensor([[ 0.0144, -0.1026, -0.3501,  0.0326,  0.1877],
        [-0.0304, -0.1648, -0.3632,  0.0139,  0.2439],
        [-0.0300, -0.1547, -0.2732,  0.0436,  0.2467],
        [ 0.0049, -0.1355, -0.1885,  0.0037,  0.1674],
        [-0.0466, -0.1478, -0.2062,  0.1017,  0.0615],
        [-0.0246, -0.1280, -0.2590,  0.0841,  0.0881],
        [-0.0286, -0.1454, -0.2947,  0.0840,  0.0900],
        [-0.0142, -0.1794, -0.2